# Chapter 2: Data Handling, Joins, and Missing Data

**Foundations of Data Science**

## In this lesson

1. How last chapter's table was actually built: four files, one key, three kinds of join
2. Who disappears when you join, and why that matters
3. A wrangling checklist: keys, codes, names, types
4. Why values are missing: three mechanisms, with NHANES examples
5. What to do about it: four strategies and what each one assumes
6. Watching a strategy change an answer

## The one idea

**Every join and every missing value is a decision, and the decision has to be written down.** Last chapter we read `data/processed/nhanes.parquet` and took it as given. Today we rebuild it from the raw files and look at every choice along the way. The clinic's question is the same (screen all adults from 35, or only overweight ones?); what changes is that we can now say how much our answer depends on how we handled the data.

## Setup

<!-- nb-links -->
[Download this notebook](chapter_02_data_handling.ipynb?download=1){download="chapter_02_data_handling.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/chapter_02/chapter_02_data_handling.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/chapter_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("chapter_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this chapter's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Four files, one key

NHANES is released as separate files per *component*. The four we need:

| File | Component | What's in it |
|---|---|---|
| `DEMO_J` | Demographics | everyone interviewed: age, sex, race/ethnicity, income, survey weights |
| `BMX_J` | Body measures | weight, height, BMI, waist, for people who came to the mobile exam center |
| `BPX_J` | Blood pressure | up to four readings, age 8 and over, exam center only |
| `DIQ_J` | Diabetes questionnaire | "has a doctor ever told you that you have diabetes?", age 1 and over |

They share one column, `SEQN`, the participant id. Everything today hinges on that column, so the first thing to check is that it uniquely identifies a row in each file. If it doesn't, a join will silently multiply rows.

In [ ]:
import pandas as pd

from core.datasets import load_raw

demo = load_raw("nhanes_demo")
bmx = load_raw("nhanes_bmx")
bpx = load_raw("nhanes_bpx")
diq = load_raw("nhanes_diq")

# Row counts and key uniqueness. An assert fails loudly, which is what you want from a data check.
for name, df in [("demo", demo), ("bmx", bmx), ("bpx", bpx), ("diq", diq)]:
    assert df["SEQN"].is_unique, f"{name}: SEQN is not unique"
    print(f"{name:<5} {df.shape[0]:>6,} rows  {df.shape[1]:>3} columns  SEQN unique: True")

In [ ]:
from eda_toolkit import dataframe_profiler

# Profile the raw body-measures file before joining. Note the null percentages on each measurement
# and the comment-code columns (BMIWT, BMIHT ...) that explain *why* a measurement is missing.
dataframe_profiler(bmx, return_df=True)

## 2. Three kinds of join, and who disappears

The counts differ: 9,254 people were interviewed, 8,704 showed up for the exam, 8,897 answered the diabetes question. A join has to decide what to do with people who are in one file and not the other.

- **inner**: keep only ids present in *both* tables. Anyone missing from either side is dropped.
- **left**: keep every row of the left table; where the right table has no match, fill with missing values.
- **outer**: keep every id from either side, filling missing on both.

Plain rule: `inner` throws people away, `left` keeps them and marks the gap as missing. Neither is "right"; the question is whether the people you'd throw away differ from the people you'd keep. If they do, an inner join changes who your dataset is about.

In [ ]:
# Same two tables, two joins. Watch the row count.
inner = demo.merge(bmx, on="SEQN", how="inner")
left = demo.merge(bmx, on="SEQN", how="left")
print("inner:", inner.shape[0], "  left:", left.shape[0], "  demo alone:", demo.shape[0])

# Who is in demo but not in bmx? indicator=True adds a column saying where each row came from.
check = demo.merge(bmx[["SEQN"]], on="SEQN", how="left", indicator=True)
no_exam = check[check["_merge"] == "left_only"]
print("interviewed but not examined:", no_exam.shape[0])

In [ ]:
# Do the 550 non-examined people look like everyone else? Compare age distributions.
# If they don't, an inner join quietly shifts the age mix of the dataset.
compare = pd.DataFrame({
    "examined": check.loc[check["_merge"] == "both", "RIDAGEYR"].describe(),
    "not_examined": no_exam["RIDAGEYR"].describe(),
}).round(1)
compare

The 550 people who skipped the exam are, if anything, slightly younger (median 26 vs 31), with a wider spread. On age alone, an inner join wouldn't distort much. That is a reassuring check, and it is still only one check: they might differ on income, health, or something we didn't measure. `preprocessing.py` uses `left` joins for the exam components anyway, for two reasons: it keeps everyone interviewed, and it leaves the gap visible as a missing value instead of silently deleting 550 people. You can always drop them later, on purpose, with a reason.

The questionnaire is different: the diabetes question *defines* the cohort for our question. Someone who never answered it can't be a case or a non-case, so an `inner` join there is the honest choice, and it has to be said out loud.

In [ ]:
# Rebuild the merged table step by step, the same way preprocessing.py does it.
# Select columns first: carrying 100+ columns through a merge is slow and makes mistakes harder to see.
demo_cols = ["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "INDFMPIR", "WTMEC2YR"]
bmx_cols = ["SEQN", "BMXWT", "BMXHT", "BMXBMI", "BMXWAIST"]
bpx_cols = ["SEQN", "BPXSY1", "BPXDI1"]
diq_cols = ["SEQN", "DIQ010"]

merged = (
    demo[demo_cols]
    .merge(bmx[bmx_cols], on="SEQN", how="left")   # keep everyone interviewed
    .merge(bpx[bpx_cols], on="SEQN", how="left")   # keep everyone interviewed
    .merge(diq[diq_cols], on="SEQN", how="inner")  # must have answered the diabetes question
)
print(merged.shape)
merged.head()

## 3. A wrangling checklist

Before analysis, four mechanical things, every time:

1. **Key**: unique on each side, same dtype on each side (an int `SEQN` won't match a float `SEQN`).
2. **Codes**: survey data stores answers as numbers. `DIQ010`: 1 yes, 2 no, 3 borderline, 7 refused, 9 don't know. Treating 7 and 9 as data is a classic error. Look up the codebook; never guess.
3. **Names**: `BMXBMI` is fine for the CDC and useless in a plot legend. Rename once, early, in one place.
4. **Types**: categorical codes stored as floats (`1.0`, `2.0`) should become integers or labeled categories so they can't be averaged by accident.
5. **Impossible values**: a BMI of 86 or a systolic pressure of 228 is either a real extreme or a data-entry error. Find them, look at the rows, decide; never delete silently.

Note what we do with 7 and 9 below: we *drop* those rows, because "refused" is not a third diabetes status. That removes 4 people. It's a decision; it's small; it's still written down.

In [ ]:
# 2. Codes: what did people actually answer?
print(merged["DIQ010"].value_counts(dropna=False).sort_index())

# Keep yes / no / borderline; recode to a 0/1 target where borderline counts as no (another decision to state).
merged = merged[merged["DIQ010"].isin([1, 2, 3])].copy()
merged["diabetes"] = (merged["DIQ010"] == 1).astype(int)

# 3. Names, 4. Types
merged = merged.rename(columns={
    "RIAGENDR": "sex", "RIDAGEYR": "age", "RIDRETH3": "race_eth", "INDFMPIR": "income_ratio",
    "WTMEC2YR": "weight_mec", "BMXWT": "weight_kg", "BMXHT": "height_cm", "BMXBMI": "bmi",
    "BMXWAIST": "waist_cm", "BPXSY1": "sbp", "BPXDI1": "dbp",
}).drop(columns="DIQ010")
merged["sex"] = merged["sex"].map({1: "male", 2: "female"}).astype("category")

print(merged.shape)
merged.dtypes

In [ ]:
from eda_toolkit import detect_outliers

# 5. Impossible values. IQR rule: flag anything beyond 1.5 x the interquartile range from the quartiles.
# The summary gives the count, percent, and the cutoffs, so you can say in words what "outlier" meant.
summary, mask = detect_outliers(merged, features=["bmi", "sbp", "dbp"], method="iqr", return_mask=True)
print(summary)

# Look before you act. Are the flagged BMIs errors, or the heaviest people in the sample?
merged.loc[mask["bmi"]].sort_values("bmi", ascending=False)[
    ["SEQN", "age", "sex", "bmi", "sbp", "dbp", "diabetes"]
].head(8)

These are extreme, not impossible: the heaviest adults in a national sample of 9,000 will have BMIs in the 50s and 60s, and some of them report diabetes. The IQR rule flags the tails of a skewed distribution, which is not the same as finding errors. The decision here is to **keep them** and note that BMI is right-skewed, which matters for the statistics recap next chapter. The one thing we'd remove is a value outside what a human body can produce; there are none.

One more thing in that table, and it previews section 4: several of the very heaviest people have *no blood pressure reading*. That is not a coincidence. A standard cuff doesn't fit a very large arm, so the reading is more likely to be missing exactly when it would be most interesting. Missingness that depends on the thing you're trying to measure has a name, and it's the hardest kind.

## 4. Why is it missing?

Statisticians sort missing values into three mechanisms. The names are awkward; the ideas are simple.

- **MCAR, missing completely at random**: the gap has nothing to do with anything. A dropped test tube. Rare in practice.
- **MAR, missing at random** (badly named): whether a value is missing depends on *other things you observed*, not on the value itself. Blood pressure is missing for children because NHANES doesn't measure it under age 8. Given age, the gap is predictable.
- **MNAR, missing not at random**: whether it's missing depends on the value itself. People with very high incomes decline the income question more often. The gap carries information about the thing you can't see.

Why it matters: under MCAR, dropping rows loses precision but not truth. Under MAR, you can recover the truth if you use the other observed variables. Under MNAR, nothing you do with the observed data fully fixes it; you can only bound the damage and say so.

You can't prove which mechanism holds from the data alone. You can look for patterns that rule MCAR out, which is what the next two cells do.

In [ ]:
# Missingness by age band. If a column is missing almost entirely in one band, that's a design rule, not an accident.
age_bins = [0, 8, 20, 40, 60, 81]
age_labels = ["0-7", "8-19", "20-39", "40-59", "60+"]
merged["age_band"] = pd.cut(merged["age"], bins=age_bins, labels=age_labels, right=False)

missing_by_age = (
    merged.groupby("age_band", observed=True)[["bmi", "sbp", "income_ratio"]]
    .apply(lambda g: g.isna().mean())
    .round(3)
)
missing_by_age

In [ ]:
# Is missing income related to anything else we observe? Compare people with and without income_ratio.
# A difference here rules out MCAR for that column.
merged["income_missing"] = merged["income_ratio"].isna()
(
    merged.groupby("income_missing")[["age", "bmi", "diabetes"]]
    .mean()
    .round(3)
    .rename(index={False: "income present", True: "income missing"})
)

In [ ]:
from eda_toolkit import contingency_table

# Is the income gap evenly spread across age bands? A contingency table with counts and percentages.
contingency_table(merged, cols=["age_band", "income_missing"])

Read the two tables. Blood pressure is 100% missing under age 8 (design: not measured) and about 15% missing after, which is the exam no-shows plus readings that couldn't be taken. The first part is MAR given age and fixed by restricting to the right ages; the second part needs the same scrutiny as BMI. BMI is missing 5 to 9% from age 8 up, and 27% under 8 because NHANES doesn't compute BMI for infants under 2: again design. Income is missing 12 to 15% in every band, and the people who skipped it are modestly older, heavier, and slightly more likely to report diabetes. Small differences, but not zero, so MCAR is out for income; MAR at best, and the usual story (high earners decline) suggests some MNAR.

## 5. What to do about it

Four strategies, from crude to careful. Each one is a bet about the mechanism.

| Strategy | What it does | What it assumes | When it's fine |
|---|---|---|---|
| **Complete case** (drop rows) | keep only rows with no gaps in the columns you use | MCAR, or at least that the dropped rows don't differ on what you care about | small, random gaps |
| **Constant / median fill** | replace every gap with one number | the gap is unrelated to everything; shrinks variance, distorts relationships | a feature you barely use |
| **Group fill** | fill with the median of a group that predicts the value (e.g. age band and sex) | MAR given the grouping variables | gaps predictable from a few observed columns |
| **Model-based** (KNN, iterative imputation, multiple imputation) | predict each gap from all other columns, ideally several times with noise | MAR given the full set of observed columns | the serious option when the variable matters |

Two rules that hold regardless of strategy:

1. **Add a missingness indicator** (a 0/1 column, `bmi_missing`) when the gap might be informative. The model can then learn that "was missing" means something.
2. **Never impute the outcome.** If diabetes status is missing, that row is not a case or a non-case; it's out.

We'll use model-based imputation inside scikit-learn pipelines in chapter 7, where it belongs (fit on training data only, applied to test data). Today: watch the first three change an answer.

In [ ]:
# Three ways to handle missing BMI, and what each does to "mean BMI by diabetes status" among adults.
adults = merged[merged["age"] >= 20].copy()
print("adults:", adults.shape[0], " missing bmi:", adults["bmi"].isna().sum())

# (a) complete case
a = adults.dropna(subset=["bmi"])

# (b) overall median fill
b = adults.copy()
b["bmi"] = b["bmi"].fillna(b["bmi"].median())

# (c) group median fill: median BMI within age band and sex.
#     groupby_imputer does the transform("median") + fillna in one call and falls back to the global
#     median if a group has no observed values at all (an edge case the hand-rolled version ignores).
from eda_toolkit import groupby_imputer

c = groupby_imputer(adults, impute_col="bmi", by=["age_band", "sex"], stat="median", as_new_col=False)

results = pd.DataFrame({
    "complete_case": a.groupby("diabetes")["bmi"].mean(),
    "median_fill": b.groupby("diabetes")["bmi"].mean(),
    "group_median_fill": c.groupby("diabetes")["bmi"].mean(),
}).round(2)
results.index = ["no diabetes", "diabetes"]
results

In [ ]:
# The same three choices, applied to the plot from chapter 1: diabetes prevalence by age band and BMI category.
# Does the clinic's answer move?
def prevalence_table(df: pd.DataFrame) -> pd.Series:
    df = df.copy()
    df["bmi_cat"] = pd.cut(df["bmi"], bins=[0, 25, 30, 100], labels=["normal", "overweight", "obese"], right=False)
    return df.dropna(subset=["bmi_cat"]).groupby(["age_band", "bmi_cat"], observed=True)["diabetes"].mean()

pd.DataFrame({
    "complete_case": prevalence_table(a),
    "median_fill": prevalence_table(b),
    "group_median_fill": prevalence_table(c),
}).round(3)

### What moved, and what didn't

The mean-BMI table barely changes across strategies: 7% missing, roughly at random with respect to BMI, so any sensible fill lands within a quarter of a BMI point. The prevalence table moves even less, under half a percentage point anywhere. The only visible shift is in the overweight bucket, because the median adult BMI is about 28, so a median fill drops all 394 imputed adults into that one group and dilutes its prevalence slightly. Group-median fill does the same thing more quietly.

The lesson isn't "imputation is dangerous." It's that **you have to run the comparison to know**. If three reasonable strategies give the same answer, say so and move on. If they don't, the data are telling you the answer depends on an assumption you can't check, and that goes in the write-up.

## 6. Writing it down

Everything above is now encoded in two places:

- `preprocessing.py`: the joins, the code recoding, the renames. Anyone can read it and reproduce `data/processed/nhanes.parquet`.
- This notebook: the *reasons*. Why left joins, why inner on the questionnaire, why 7 and 9 are dropped, why borderline counts as no.

A script without reasons is a black box. Reasons without a script are a story nobody can check. Keep both.

## Looking ahead
- **Reading**: VanderPlas chapter 3, sections on merging and handling missing data. Bruce & Bruce, chapter 1 (estimates of location and variability) for next chapter's statistics recap.
- **Project**: teams should be formed. One-page problem statement is due chapter 4.